# 05 · Cenários de incidente (v3)
**CRISP-DM:** Preparação dos Dados (gabarito de avaliação)

Cada **cenário** simula um incidente: uma degradação começa em t₀ e o incidente é registrado em t = t₀ + atraso. A pergunta é respondida **no instante t**, olhando só para trás.

| Motivo | Abrangência | Baixa | Média | Alta | Muito alta |
|---|---|---|---|---|---|
| Erro de negócio (4xx) | generalizado | 5% | 15% | 40% | — |
| Erro de negócio (4xx) | 1 consumidor | 30% | 60% | 90% | — |
| Erro de sistema (5xx) | generalizado | 5% | 15% | 40% | — |
| Erro de sistema (5xx) | 1 consumidor | 30% | 60% | 90% | — |
| Latência | generalizada | ×1,5 | ×2,5 | ×4 | **×10** |

**Latência ×10 (novo na v3).** Como a latência vem de médias por minuto, poucas requisições lentas elevam muito o valor: no histórico, o p95 horário passa de 10× a mediana do seu contexto em ~0,5% das horas normais (notebook 06 mostra a distribuição). Degradações de latência relevantes para um incidente tendem a ser desse tamanho.

- **Meses de teste:** agosto e 01–20/09. Entram os recursos com ao menos 28 dias de dados no mês anterior (5 recursos que começam em 16/07 só entram em setembro).
- **Atraso** entre o início da degradação e o registro: 15, 30, 60 ou 120 min; **5 repetições** por combinação; o mesmo número de **consultas de controle** (sem falha).
- O instante do registro é sorteado dentro do mês de teste, a partir de 4 h após o início dos dados do recurso naquele mês.

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v10) carregado · /home/wsl/gh/mestrado/claude9/v10


In [2]:
brutos = pd.read_parquet(RESULTADOS / "02_brutos_consolidados.parquet")
B = Brutos(brutos)
eleg = elegiveis(B)
eleg.assign(recurso=eleg["recurso"].map(rot)).pivot(index="recurso", columns="mes", values=["dias_hist_1m", "elegivel"]).sort_index()

dias_hist_1m     elegivel      
mes              ago set      ago   set
recurso                                
E01               31  31     True  True
E02               16  31    False  True
E03               16  31    False  True
E04               31  31     True  True
E05               31  31     True  True
E06               16  31    False  True
E07               31  31     True  True
E08               31  31     True  True
E09               31  31     True  True
E10               16  31    False  True
E11               16  31    False  True
E12               31  31     True  True

In [3]:
cenarios = gerar_cenarios(eleg)
cenarios.to_csv(RESULTADOS / "05_cenarios.csv", index=False)
eleg.to_csv(RESULTADOS / "05_elegiveis.csv", index=False)
print(cenarios.groupby(["mes", "tipo"]).size().unstack().to_dict())
cenarios.assign(recurso=cenarios["recurso"].map(rot)).query("tipo == 'incidente'").groupby(["recurso", "mes"]).size().unstack()

{'controle': {'ago': 2240, 'set': 3840}, 'incidente': {'ago': 2240, 'set': 3840}}


mes,ago,set
recurso,,
E01,320.0,320.0
E02,NaN,320.0
E03,NaN,320.0
E04,320.0,320.0
E05,320.0,320.0
E06,NaN,320.0
E07,320.0,320.0
E08,320.0,320.0
E09,320.0,320.0


## Exemplo: efeito de um cenário nas faixas antes do registro

In [4]:
ex = cenarios[(cenarios["tipo"] == "incidente") & (cenarios["motivo"] == "5xx_geral") & (cenarios["intensidade"] == "media")
              & (cenarios["atraso"] == 30)].iloc[0]
sem = ex.copy()
sem["tipo"] = "controle"
com_falha = pd.DataFrame(periodos_do_cenario(B, ex))
sem_falha = pd.DataFrame(periodos_do_cenario(B, sem))
cols = ["periodo", "lo", "hi", "vol", "tx_4xx", "tx_5xx", "lat_p95", "n_cli_5xx"]
print("recurso", rot(ex["recurso"]), "· registro", ex["t_registro"], "· início real 30 min antes")
display(sem_falha[cols].round(3).rename(columns=lambda c: c + " (sem)" if c not in ("periodo", "lo", "hi") else c)
        .join(com_falha[["tx_5xx", "n_cli_5xx"]].round(3).rename(columns=lambda c: c + " (com)")))

recurso E12 · registro 2026-08-14 17:52:00 · início real 30 min antes


,periodo,lo,hi,vol (sem),tx_4xx (sem),tx_5xx (sem),lat_p95 (sem),n_cli_5xx (sem),tx_5xx (com),n_cli_5xx (com)
0,L,0,15,175,0.000,0.000,240.0,0,0.166,10
1,L,0,30,465,0.000,0.000,205.0,0,0.168,22
2,L,0,60,1243,0.013,0.002,364.0,2,0.065,23
3,banda,0,15,175,0.000,0.000,240.0,0,0.166,10
4,banda,15,30,290,0.000,0.000,205.0,0,0.169,17
5,banda,30,60,778,0.021,0.004,402.0,2,0.004,2
6,banda,60,120,2131,0.002,0.004,188.0,1,0.004,1
7,banda,120,180,1663,0.000,0.008,387.0,3,0.008,3
8,banda,180,240,1993,0.002,0.005,197.0,6,0.005,6
